#### setup

In [ ]:
import json
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from latent_switch.lens import script_mask

MODEL = "Qwen/Qwen3-4B"
TAG = MODEL.split("/")[-1].lower()
device = "mps" if torch.backends.mps.is_available() else "cpu"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to(device)
model.eval()

pilot = pd.read_csv("../data/prompts/b1_pilot.csv")
math = pilot[pilot.source == "mclm_math100"].reset_index(drop=True)      # 20 items x ne/hi/ru = 60 prompts
PROMPTS = pilot.set_index(["source", "item", "lang"])["prompt"]
GEN_DIR, ACT_DIR = Path("../data/generations"), Path("../data/activations")
ACT_DIR.mkdir(parents=True, exist_ok=True)

THINK_END = tok.convert_tokens_to_ids("</think>")
V = model.config.vocab_size                       # 151936 logit columns (tokenizer has 151669; the rest are padding)
X_SCRIPT = {"ne": "Deva", "hi": "Deva", "ru": "Cyrl"}
MASKS = {s: script_mask(tok, s, V) for s in ["Deva", "Cyrl", "Latn"]}
print(device, TAG, {s: int(m.sum()) for s, m in MASKS.items()})

import pickle

def cached(path, fn, rerun=False):
    """Load fn()'s result from `path` if it was saved before; else run fn() and save it. .csv = DataFrame, else pickle.
    rerun=True recomputes on purpose. The cache can't see code changes: rerun (or delete the file) after editing fn."""
    if path.exists() and not rerun:
        print(f"loaded {path.name}  (rerun=True to recompute)")
        return pd.read_csv(path) if path.suffix == ".csv" else pickle.loads(path.read_bytes())
    out = fn()
    if path.suffix == ".csv":
        out.to_csv(path, index=False)
    else:
        path.write_bytes(pickle.dumps(out))
    print(f"saved {path.name}")
    return out

readout at the two template decision points

In [ ]:
def chat_ids(prompt, thinking, suffix=""):
    """Chat-template ids for one user turn, plus optional text we write as the start of the assistant turn. [1, seq]"""
    text = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                   add_generation_prompt=True, enable_thinking=thinking)
    return tok(text + suffix, return_tensors="pt", add_special_tokens=False)["input_ids"]

# Both end right where the model writes its first word:
#   think_start : thinking on,  ...assistant\n<think>\n                -> first word of the trace
#   answer_start: thinking off, ...assistant\n<think>\n\n</think>\n\n  -> first word of the answer
WHERE = {"think_start": dict(thinking=True, suffix="<think>\n"), "answer_start": dict(thinking=False)}

@torch.no_grad()
def readout(ids):
    """One forward pass. Returns next-token probs [V] and the last position's hidden state at every layer [n_layers+1, d_model]."""
    out = model(input_ids=ids.to(device), output_hidden_states=True)
    probs = out.logits[0, -1].float().softmax(dim=-1).cpu()
    resid = torch.stack([h[0, -1] for h in out.hidden_states]).cpu()
    return probs, resid

def script_row(probs, lang):
    pX, pL = probs[MASKS[X_SCRIPT[lang]]].sum().item(), probs[MASKS["Latn"]].sum().item()
    top = probs.argmax().item()
    return dict(pX=pX, pL=pL, p_other=1 - pX - pL, x_vs_L=pX / (pX + pL) if pX + pL else float("nan"),
                top_tok=tok.decode([top]), p_top=probs[top].item())

def run_decision():
    rows, resid_all = [], {}
    for _, r in math.iterrows():
        for where, kw in WHERE.items():
            probs, resid = readout(chat_ids(r["prompt"], **kw))
            rows.append(dict(item=int(r["item"]), lang=r["lang"], where=where, **script_row(probs, r["lang"])))
            resid_all[(int(r["item"]), r["lang"], where)] = resid
    return pd.DataFrame(rows), resid_all

dec, RESID = cached(ACT_DIR / f"c2_decision_{TAG}.pkl", run_decision)    # RESID: {(item, lang, where): [n_layers+1, d_model]}
display(dec.groupby(["lang", "where"], sort=False)[["pX", "pL", "p_other", "x_vs_L"]].agg(["mean", "min", "max"]).round(3))
display(dec.groupby(["lang", "where"], sort=False)["top_tok"].agg(lambda s: s.value_counts().head(3).to_dict()))

In [ ]:
import regex
def has_letter(tid):
    return tid not in tok.added_tokens_decoder and regex.search(r"\p{L}", tok.decode([tid])) is not None

force = [json.loads(l) for l in open(GEN_DIR / f"b1_force_test_{TAG}.jsonl")]
brow = []
for r in force:
    if r["cond"] != "natural" or THINK_END not in r["gen_ids"]:
        continue
    g = r["gen_ids"]
    k = g.index(THINK_END) + 1
    while k < len(g) and not has_letter(g[k]):   # skip "\n\n", "###", "**": stop right before the first word token
        k += 1
    ids = torch.cat([chat_ids(PROMPTS[(r["source"], r["item"], r["lang"])], thinking=True),
                     torch.tensor([g[:k]])], dim=1)            # prompt + the model's own <think>...trace...</think>\n\n###
    probs, _ = readout(ids)
    brow.append(dict(item=r["item"], lang=r["lang"], n_ctx=ids.shape[1], **script_row(probs, r["lang"]),
                     actual=tok.decode([g[k]]), answer_head=r["answer_text"].strip()[:40]))
display(pd.DataFrame(brow).round(3))

logit lens at the decision points.. start and end of `<think>`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from latent_switch.plotting import _band
from latent_switch.stats import mean_ci

MASKS_DEV = {s: m.to(device) for s, m in MASKS.items()}
MASKS_DEV["Hans"] = script_mask(tok, "Hans", V).to(device)      # Chinese: Qwen's middle layers often decode to it

@torch.no_grad()
def lens_script_mass(resid, lang):
    """resid [n_prompts, n_layers+1, d_model] -> (pX, pL, pH), each [n_prompts, n_layers+1]:
    script mass (X, Latin, Han) of the next token if we stopped the model at each layer and read it out."""
    h = resid.to(device)
    h = torch.cat([model.model.norm(h[:, :-1]), h[:, -1:]], dim=1)      # last entry is already normed (transformers 5.x)
    probs = model.lm_head(h).float().softmax(dim=-1)                      # [n, n_layers+1, V]
    pX = probs[..., MASKS_DEV[X_SCRIPT[lang]]].sum(-1)
    pL = probs[..., MASKS_DEV["Latn"]].sum(-1)
    pH = probs[..., MASKS_DEV["Hans"]].sum(-1)
    return pX.cpu().numpy(), pL.cpu().numpy(), pH.cpu().numpy()

ITEMS = sorted(math["item"].unique())
LENS = {}
for lang in ["ne", "hi", "ru"]:
    for where in WHERE:
        resid = torch.stack([RESID[(i, lang, where)] for i in ITEMS])     # [20, n_layers+1, d_model]
        LENS[(lang, where)] = lens_script_mass(resid, lang)

# sanity: the last layer of the lens must equal the real output measured in cell 2
for (lang, where), (pX, pL, pH) in LENS.items():
    real = dec[(dec["lang"] == lang) & (dec["where"] == where)].set_index("item").loc[ITEMS, "pX"].to_numpy()
    assert np.allclose(pX[:, -1], real, atol=2e-2), (lang, where)

layers = np.arange(LENS[("ne", "think_start")][0].shape[1])
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, lang in zip(axes, ["ne", "hi", "ru"]):
    for where, ls in [("think_start", "-"), ("answer_start", "--")]:
        pX, pL, pH = LENS[(lang, where)]
        _band(ax, layers, mean_ci(pX), "C1", ls, f"{X_SCRIPT[lang]} | {where}")
        _band(ax, layers, mean_ci(pL), "C0", ls, f"Latin | {where}")
        _band(ax, layers, mean_ci(pH), "C2", ls, f"Han | {where}")
    ax.set(title=f"{lang}: next-token script mass by layer (n={len(ITEMS)})", xlabel="layer")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("probability")
fig.suptitle(f"{TAG} · logit lens at the first word: inside <think> (solid) vs answer, thinking off (dashed)")
fig.tight_layout()

checking that dip in layer ~24 ish

In [ ]:
LAYERS = [20, 22, 23, 24, 25, 27, 29, 31, 32, 33, 34, 35, 36]

@torch.no_grad()
def lens_top(lang, where, layers=LAYERS, k=4):
    """Lens readout at chosen layers, averaged over the 20 items.
    Returns {layer: (top-k [(token, prob)], Han mass)}."""
    resid = torch.stack([RESID[(i, lang, where)] for i in ITEMS]).to(device)    # [20, n_layers+1, d_model]
    h = torch.cat([model.model.norm(resid[:, :-1]), resid[:, -1:]], dim=1)     # same lens as the plot
    probs = model.lm_head(h[:, layers]).float().softmax(dim=-1).mean(dim=0)    # [len(layers), V], mean over items
    p, idx = probs.topk(k, dim=-1)                                              # both [len(layers), k]
    han = probs[:, MASKS_DEV["Hans"]].sum(dim=-1)                               # [len(layers)]
    return {L: ([(tok.decode([t]), v) for t, v in zip(idx[j].tolist(), p[j].tolist())], han[j].item())
            for j, L in enumerate(layers)}

def fmt(top, han):
    return f"Han {han:.2f} | " + "  ".join(f"{t!r} {v:.2f}" for t, v in top)

with pd.option_context("display.max_colwidth", None):
    for lang in ["ne", "hi", "ru"]:
        res = {where: lens_top(lang, where) for where in WHERE}
        tab = pd.DataFrame({where: {L: fmt(*res[where][L]) for L in LAYERS} for where in WHERE})
        tab.index.name = f"{lang} layer"
        display(tab)

past the "Okay" reflex: lens at the content words of the first 40 trace tokens (greedy). Caveat: by then the context is already in the trace language, so this mixes mode with persistence.

In [ ]:
import regex

K = 40                                                    # trace tokens to generate and read
HAS_LETTER = regex.compile(r"\p{L}")

@torch.no_grad()
def greedy_trace(prompt, k=K):
    """Greedy start of the thinking trace. Returns (prompt_len, ids [prompt_len + k])."""
    ids = chat_ids(prompt, **WHERE["think_start"]).to(device)
    out = model.generate(ids, attention_mask=torch.ones_like(ids), max_new_tokens=k, do_sample=False,
                         temperature=None, top_p=None, top_k=None)
    return ids.shape[1], out[0].cpu()

@torch.no_grad()
def lens_at(ids, pos, lang):
    """Lens script mass at positions `pos` (each reads the NEXT token). ids [seq].
    Returns pX, pL, pH [n_layers+1, len(pos)] and the final-layer argmax ids [len(pos)]."""
    out = model(input_ids=ids[None].to(device), output_hidden_states=True)
    pX, pL, pH = [], [], []
    for l, hs in enumerate(out.hidden_states):            # one layer at a time: [len(pos), V] instead of [37, len(pos), V]
        h = hs[0, pos]                                     # [len(pos), d_model]
        h = h if l == len(out.hidden_states) - 1 else model.model.norm(h)   # last entry is already normed
        probs = model.lm_head(h).float().softmax(dim=-1)   # [len(pos), V]
        pX.append(probs[:, MASKS_DEV[X_SCRIPT[lang]]].sum(-1))
        pL.append(probs[:, MASKS_DEV["Latn"]].sum(-1))
        pH.append(probs[:, MASKS_DEV["Hans"]].sum(-1))
    to_np = lambda xs: torch.stack(xs).cpu().numpy()
    return to_np(pX), to_np(pL), to_np(pH), probs.argmax(-1).cpu()

def run_content():
    trace, content = {}, {}
    for _, r in math.iterrows():
        lang, item = r["lang"], int(r["item"])
        n_prompt, ids = greedy_trace(r["prompt"])
        gen = ids[n_prompt:].tolist()
        # content words: trace tokens with a letter, skipping the first one ("Okay"); read at the position before each
        ts = [t for t in range(1, len(gen)) if gen[t] not in tok.added_tokens_decoder and HAS_LETTER.search(tok.decode([gen[t]]))]
        pos = [n_prompt + t - 1 for t in ts]
        pX, pL, pH, top = lens_at(ids, pos, lang)
        trace[(item, lang)] = tok.decode(gen)
        content[(item, lang)] = dict(pX=pX.mean(1), pL=pL.mean(1), pH=pH.mean(1), n=len(pos),
                                     match=(top == torch.tensor([gen[t] for t in ts])).float().mean().item())
    return trace, content

TRACE, CONTENT = cached(ACT_DIR / f"c2_content_{TAG}.pkl", run_content)

# sanity: the final-layer argmax should reproduce the greedy tokens (≈1.0; bf16 near-ties can flip a few)
print("greedy match:", round(np.mean([v["match"] for v in CONTENT.values()]), 3),
      "| content words per trace:", min(v["n"] for v in CONTENT.values()), "-", max(v["n"] for v in CONTENT.values()))
for lang in ["ne", "hi", "ru"]:
    print(f"{lang}: {TRACE[(ITEMS[0], lang)][:150]!r}")

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5), sharey=True)
for ax, lang in zip(axes, ["ne", "hi", "ru"]):
    for key, color, name in [("pX", "C1", X_SCRIPT[lang]), ("pL", "C0", "Latin"), ("pH", "C2", "Han")]:
        content = np.stack([CONTENT[(i, lang)][key] for i in ITEMS])            # [20, n_layers+1], mean over positions
        _band(ax, layers, mean_ci(content), color, "-", f"{name} | content words")
        okay = LENS[(lang, "think_start")][["pX", "pL", "pH"].index(key)]          # first token ("Okay"), for reference
        ax.plot(layers, okay.mean(0), color=color, ls=":", lw=1, label=f"{name} | first token")
    ax.set(title=f"{lang}: inside <think>, first {K} trace tokens (n={len(ITEMS)})", xlabel="layer")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_ylabel("probability")
fig.suptitle(f"{TAG} · logit lens at content words inside the trace (solid) vs the first token 'Okay' (dotted)")
fig.tight_layout()

activation patching (first causal test): same problem, same mode, swap the language. Run the ne prompt, but at one layer L overwrite the last position's state with the ru run's state (same item, same mode, same last token). The flip layer is where the output language is already fixed in that state; before it, later layers still read the language from the prompt. Compare with the lens: the lens shows X only at the last 1–2 layers.

In [ ]:
N_LAYERS = model.config.num_hidden_layers
PATCH_LAYERS = list(range(1, N_LAYERS))   # L = hidden_states index = output of block L-1. Skip L=N_LAYERS: RESID[..][-1] is already normed

@torch.no_grad()
def patched_probs(ids, L, vec):
    """Run ids, but right after block L-1 overwrite the LAST position's residual with vec [d_model]. Returns next-token probs [V]."""
    def hook(module, args, output):        # a Qwen3 decoder layer returns a plain tensor [batch, seq, d_model]
        output[0, -1] = vec                # in-place: every later block sees the new value
    handle = model.model.layers[L - 1].register_forward_hook(hook)
    try:
        logits = model(input_ids=ids.to(device), logits_to_keep=1).logits[0, -1]
    finally:
        handle.remove()                    # always remove, or the hook stays on the model for every later call
    return logits.float().softmax(-1)

# sanity: writing a run's OWN value back must change nothing (catches an off-by-one in L)
ids0 = chat_ids(PROMPTS[("mclm_math100", ITEMS[0], "ne")], **WHERE["answer_start"])
with torch.no_grad():
    clean0 = model(input_ids=ids0.to(device)).logits[0, -1].float().softmax(-1)
for L in [1, N_LAYERS // 2, N_LAYERS - 1]:
    own = RESID[(ITEMS[0], "ne", "answer_start")][L].to(device, model.dtype)
    print(f"own-value patch at L={L}: max |Δp| = {(patched_probs(ids0, L, own) - clean0).abs().max().item():.1e}")

# the script each clean run actually writes its first word in (think-mode ne/hi write English "Okay")
OUT = {(lang, where): X_SCRIPT[lang] if g["pX"].mean() > 0.5 else "Latn"
       for (lang, where), g in dec.groupby(["lang", "where"])}
PAIRS = [("ne", "ru"), ("hi", "ru"), ("ru", "ne")]       # (language of the run, language whose state we patch in)

def run_patch():
    rows = []
    for where in WHERE:
        for run, src in PAIRS:
            for i in ITEMS:
                ids = chat_ids(PROMPTS[("mclm_math100", i, run)], **WHERE[where])
                for L in PATCH_LAYERS:
                    p = patched_probs(ids, L, RESID[(i, src, where)][L].to(device, model.dtype))
                    rows.append(dict(where=where, run=run, src=src, item=i, layer=L,
                                     p_run=p[MASKS_DEV[OUT[(run, where)]]].sum().item(),
                                     p_src=p[MASKS_DEV[OUT[(src, where)]]].sum().item(),
                                     top=tok.decode([p.argmax().item()])))
            print(where, run, "<-", src, "done")
    return pd.DataFrame(rows)

patch = cached(ACT_DIR / f"c2_patch_lang_{TAG}.csv", run_patch)

fig, axes = plt.subplots(2, 3, figsize=(18, 8), sharex=True, sharey=True)
for r, where in enumerate(WHERE):
    for c, (run, src) in enumerate(PAIRS):
        ax, P = axes[r, c], patch[(patch["where"] == where) & (patch.run == run)]
        for col, color, who in [("p_run", "C1", run), ("p_src", "C3", src)]:
            m = P.pivot(index="item", columns="layer", values=col).to_numpy()     # [n_items, n_patch_layers]
            _band(ax, PATCH_LAYERS, mean_ci(m), color, "-", f"{OUT[(who, where)]} ({who}'s clean output)")
        ax.set(title=f"{where}: {run} run, {src} state patched in")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
for ax in axes[1]:
    ax.set_xlabel("patched layer")
for ax in axes[:, 0]:
    ax.set_ylabel("first-word script mass")
fig.suptitle(f"{TAG} · patch the last position's state from the other language (same problem, same mode)")
fig.tight_layout()

# first layer where the patched-in language wins on average
flip = (patch.groupby(["where", "run", "src", "layer"])["p_src"].mean()
             .loc[lambda s: s > 0.5].reset_index().groupby(["where", "run", "src"])["layer"].min())
display(flip.rename("flip layer").to_frame())
display(patch.groupby(["where", "run", "layer"])["top"].agg(lambda s: s.value_counts().head(2).to_dict())
             .unstack([0, 1]).iloc[::3])


linear probing: why? what? -- what we saw in the logit lens -- lens only shwowed us the target language X tokens in very late layers like after ~32 ish, right? but the graph above shows that the information on those X tokens is actually encoded from about layer ~25ish. So, what we do is train a probe(classifier) to see this encoded information

this cell below is one forward pass for each of the templates given

In [ ]:
LANG_NAME = {"en": "English", "ne": "Nepali", "hi": "Hindi", "ru": "Russian"}
TEMPLATES = ["Answer in {}.", "Please respond in {}.", "Write your answer in {}.", "Reply only in {}."]

def run_probe_data():
    rows, H = [], []
    for _, r in pilot.iterrows():                                   # 40 questions x ne/hi/ru = 120 prompts
        for t, tpl in enumerate(TEMPLATES):
            for target in ["en", r["lang"]]:                        # the minimal pair
                ids = chat_ids(r["prompt"] + "\n\n" + tpl.format(LANG_NAME[target]), thinking=False)
                probs, resid = readout(ids)                         # resid [n_layers+1, d_model], last position
                rows.append(dict(source=r["source"], item=int(r["item"]), lang=r["lang"], tpl=t,
                                 target=target, **script_row(probs, r["lang"])))
                H.append(resid)
    return pd.DataFrame(rows), torch.stack(H)                      # H [960, n_layers+1, d_model], bf16

pdat, H = cached(ACT_DIR / f"c2_probe_pairs_{TAG}.pkl", run_probe_data)
pdat["group"] = pdat["source"] + ":" + pdat["item"].astype(str)   # split unit: one question, all its versions
pdat["y"] = (pdat["pX"] > pdat["pL"]).astype(int)                 # label = what the model writes: 1 = X script, 0 = Latin
pdat["ok"] = pdat[["pX", "pL"]].max(axis=1) > 0.5                 # the model clearly picks one of the two scripts
pdat["complied"] = pdat["y"] == (pdat["target"] != "en").astype(int)

print(H.shape, H.dtype)
display(pdat.groupby(["lang", "target"])[["pX", "pL", "ok", "complied"]].mean().round(3))
display(pdat.groupby(["lang", "target"])["top_tok"].agg(lambda s: s.value_counts().head(4).to_dict()))

linear probing per layer

In [ ]:
import numpy as np
import torch.nn.functional as F

def fit_probe(X, y, lam=1e-3, steps=200, lr=1e-2):
    """Logistic regression in plain torch. X [n, d] float32, y [n] float 0/1. Returns predict(Xnew) -> logits [m]."""
    mu, sd = X.mean(0), X.std(0) + 1e-6                  # standardize with TRAIN statistics only
    Z = (X - mu) / sd
    w = torch.zeros(X.shape[1], requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(Z @ w + b, y) + lam * (w ** 2).sum()
        loss.backward()
        opt.step()
    w, b = w.detach(), b.detach()
    return lambda Xnew: ((Xnew - mu) / sd) @ w + b

def balanced_acc(logits, y):
    """Mean of the accuracy on each class, so 0.5 = chance even when one class is more common."""
    pred = (logits > 0).float()
    return 0.5 * ((pred[y == 1] == 1).float().mean() + (pred[y == 0] == 0).float().mean()).item()

def group_folds(groups, k=5, seed=0):
    """Put each question in one of k folds, so no question is in both train and test."""
    uniq = np.random.default_rng(seed).permutation(sorted(set(groups)))
    fold_of = {g: i % k for i, g in enumerate(uniq)}
    return np.array([fold_of[g] for g in groups])

K = 5
def run_probe():
    rows = []
    for lang in ["ne", "hi", "ru"]:
        sub = pdat[(pdat["lang"] == lang) & pdat["ok"]]
        idx = torch.tensor(sub.index.to_numpy())                         # row numbers into H
        y = torch.tensor(sub["y"].to_numpy(), dtype=torch.float32)
        obeyed = torch.tensor(sub["complied"].to_numpy())
        fold = torch.tensor(group_folds(sub["group"].tolist(), K))
        for L in range(H.shape[1]):
            X = H[idx, L].float()                                         # [n, d_model], bf16 -> float32
            for f in range(K):
                te = fold == f
                # (a) normal probe: train on all rows of the other folds, label = what the model wrote
                acc = balanced_acc(fit_probe(X[~te], y[~te])(X[te]), y[te])
                # (b) instruction or plan? train only on rows where the model obeyed (label = instruction),
                #     test on held-out rows where it ignored "Answer in English" and wrote X anyway
                test_dis = te & ~obeyed
                follows = float("nan")
                if test_dis.any():
                    pred = fit_probe(X[~te & obeyed], y[~te & obeyed])(X[test_dis]) > 0
                    follows = pred.float().mean().item()                  # 1 = probe says X, like the model's output
                rows.append(dict(lang=lang, layer=L, fold=f, acc=acc, follows=follows, n_dis=int(test_dis.sum())))
    return pd.DataFrame(rows)

probe_res = cached(ACT_DIR / f"c2_probe_res_{TAG}.csv", run_probe)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for lang, c in zip(["ne", "hi", "ru"], ["C0", "C1", "C2"]):
    d = probe_res[probe_res["lang"] == lang]
    acc = d.groupby("layer")["acc"].agg(["mean", "min", "max"])
    axes[0].plot(acc.index, acc["mean"], color=c, label=lang)
    axes[0].fill_between(acc.index, acc["min"], acc["max"], color=c, alpha=0.2)
    dd = d.dropna(subset=["follows"])
    fol = (dd["follows"] * dd["n_dis"]).groupby(dd["layer"]).sum() / dd.groupby("layer")["n_dis"].sum()
    n = int(dd.loc[dd["layer"] == 0, "n_dis"].sum())
    axes[1].plot(fol.index, fol.values, color=c, ls="--" if n < 20 else "-", label=f"{lang} (n={n})")
for ax in axes:
    ax.axvline(25, color="gray", ls=":", label="patching lock (L=25)")
    ax.set(xlabel="layer")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
axes[0].set_title("held-out balanced accuracy (band = min-max over 5 folds)")
axes[1].set_title("rows where the model ignored 'Answer in English':\nfraction the probe calls X (= follows the actual output)")
fig.suptitle(f"{TAG} · linear probe at the answer start, trained within one prompt language")
fig.tight_layout()

In [ ]:
def run_transfer():
    """Train on the instruction pairs, test on the natural decision points (no instruction).
    Each decision item is scored only by probes that never saw that question."""
    rows = []
    for lang in ["ne", "hi", "ru"]:
        sub = pdat[(pdat["lang"] == lang) & pdat["ok"]]
        idx = torch.tensor(sub.index.to_numpy())
        y = torch.tensor(sub["y"].to_numpy(), dtype=torch.float32)
        groups = sub["group"].tolist()
        fold = torch.tensor(group_folds(groups, K))
        fold_of = dict(zip(groups, fold.tolist()))                        # question -> its fold
        keys = [k for k in RESID if k[1] == lang]                         # (item, lang, where): 20 items x 2 positions
        kfold = torch.tensor([fold_of[f"mclm_math100:{k[0]}"] for k in keys])
        R = torch.stack([RESID[k] for k in keys])                         # [40, n_layers+1, d_model]
        for L in range(H.shape[1]):
            X, Xd = H[idx, L].float(), R[:, L].float()
            for f in range(K):
                test = kfold == f
                if not test.any():
                    continue
                logit = fit_probe(X[fold != f], y[fold != f])(Xd[test])  # probe trained without fold f's questions
                for k, z in zip([k for k, t in zip(keys, test) if t], logit.tolist()):
                    rows.append(dict(item=k[0], lang=lang, where=k[2], layer=L, logit=z, saysX=int(z > 0)))
    return pd.DataFrame(rows)

tr = cached(ACT_DIR / f"c2_probe_transfer_{TAG}.csv", run_transfer)
display(tr.groupby(["lang", "where", "layer"])["saysX"].mean().unstack("layer")
          .loc[:, [0, 10, 20, 24, 25, 26, 28, 30, 32, 34, 36]].round(2))

see the important transformations done is each step

In [ ]:
lang, item, L = "ne", 5, 30
print(f"--- example: lang={lang}, item={item}, layer={L}")

# [1] the 8 training rows for this one question
one = pdat[(pdat["lang"] == lang) & (pdat["source"] == "mclm_math100") & (pdat["item"] == item)]
print("\n[1] training rows for this question:")
display(one[["tpl", "target", "top_tok", "pX", "pL", "y", "complied"]].round(3))

# [2] the last tokens of each context (the probe reads the LAST position)
prompt = PROMPTS[("mclm_math100", item, lang)]
ctx = {"train pair (Answer in English)": chat_ids(prompt + "\n\nAnswer in English.", thinking=False),
       "answer_start (no instruction)":   chat_ids(prompt, thinking=False),
       "think_start  (no instruction)":   chat_ids(prompt, thinking=True, suffix="<think>\n")}
print("\n[2] last 6 tokens of each context:")
for name, ids in ctx.items():
    print(f"  {name:32s} {[tok.decode([t]) for t in ids[0, -6:]]}")

# [3] training rows for this language, and which fold our question is in
def probe_for(lang, item, L):
    sub = pdat[(pdat["lang"] == lang) & pdat["ok"]]
    idx = torch.tensor(sub.index.to_numpy())
    y = torch.tensor(sub["y"].to_numpy(), dtype=torch.float32)
    groups = sub["group"].tolist()
    fold = torch.tensor(group_folds(groups, K))
    f = dict(zip(groups, fold.tolist()))[f"mclm_math100:{item}"]
    X = H[idx, L].float()
    probe = fit_probe(X[fold != f], y[fold != f])                 # never saw this question
    print(f"\n[3] {lang}: {len(sub)} rows (y=1: {int(y.sum())}, y=0: {int((1 - y).sum())}); "
          f"question {item} is in fold {f}; probe trained on {(fold != f).sum().item()} rows")
    lt, yt = probe(X[fold == f]), y[fold == f]
    print(f"    held-out training-style rows: wrote {lang} -> mean {lt[yt == 1].mean():+.1f} | "
          f"wrote en -> mean {lt[yt == 0].mean():+.1f}")
    return probe

# [4] apply to this question's no-instruction decision points; ru = control
for lg in [lang, "ru"]:
    probe = probe_for(lg, item, L)
    for where in ["answer_start", "think_start"]:
        z = probe(RESID[(item, lg, where)][L].float().unsqueeze(0)).item()
        top = dec[(dec["item"] == item) & (dec["lang"] == lg) & (dec["where"] == where)]["top_tok"].item()
        print(f"[4] {lg} {where:12s}: model writes {top!r:8s} -> probe {z:+.1f} -> says {'X' if z > 0 else 'en'}")

train the probe for the `<think_start>` position as well. before it was `<answer_start>`

In [ ]:
TEMPLATES_T = TEMPLATES + ["Think and answer in {}.", "Do all your reasoning in {}."]   # tpl 0-3 = same as the answer_start pairs

def run_probe_data_think():
    rows, H = [], []
    for _, r in pilot.iterrows():                                   # 120 prompts x 6 templates x 2 targets = 1440 passes
        for t, tpl in enumerate(TEMPLATES_T):
            for target in ["en", r["lang"]]:
                ids = chat_ids(r["prompt"] + "\n\n" + tpl.format(LANG_NAME[target]), **WHERE["think_start"])
                probs, resid = readout(ids)
                rows.append(dict(source=r["source"], item=int(r["item"]), lang=r["lang"], tpl=t,
                                 target=target, **script_row(probs, r["lang"])))
                H.append(resid)
    return pd.DataFrame(rows), torch.stack(H)                      # H [1440, n_layers+1, d_model], bf16

def add_labels(d):
    d["group"] = d["source"] + ":" + d["item"].astype(str)
    d["y"] = (d["pX"] > d["pL"]).astype(int)                       # 1 = first word in X script, 0 = Latin
    d["ok"] = d[["pX", "pL"]].max(axis=1) > 0.5
    d["complied"] = d["y"] == (d["target"] != "en").astype(int)
    return d

pdat_t, H_t = cached(ACT_DIR / f"c2_probe_pairs_think_{TAG}.pkl", run_probe_data_think)
pdat_t = add_labels(pdat_t)

print(H_t.shape, H_t.dtype)
display(pdat_t.groupby(["lang", "target"])[["pX", "pL", "ok", "complied"]].mean().round(3))
display(pdat_t.groupby(["lang", "target", "tpl"])["y"].mean().unstack("tpl").round(2))   # which templates (if any) get X thinking
display(pdat_t.groupby(["lang", "target"])["top_tok"].agg(lambda s: s.value_counts().head(4).to_dict()))


We train four probes per layer:
┌──────────────┬──────────────────────┬─────────────────────────────────────────────────────┐
│    Probe     │      Trained on      │                 Question it answers                 │
├──────────────┼──────────────────────┼─────────────────────────────────────────────────────┤
│ answer       │ answer_start only    │ Does it carry over to think_start?                  │
├──────────────┼──────────────────────┼─────────────────────────────────────────────────────┤
│ think        │ think_start only     │ Does it carry over to answer_start?                 │
├──────────────┼──────────────────────┼─────────────────────────────────────────────────────┤
│ mixed        │ both                 │ Can one probe work at both positions?               │
├──────────────┼──────────────────────┼─────────────────────────────────────────────────────┤
│ mixed_obeyed │ both, obeyed rows    │ On disobeyed rows, does it follow the request or    │
│              │ only                 │ the output?                                         │
└──────────────┴──────────────────────┴─────────────────────────────────────────────────────┘


In [ ]:
def auc(scores, y):
    """P(a random y=1 row scores higher than a random y=0 row). Ignores any shared shift of all scores."""
    pos, neg = scores[y == 1], scores[y == 0]
    if len(pos) == 0 or len(neg) == 0:
        return float("nan")
    diff = pos[:, None] - neg[None, :]                     # [n_pos, n_neg] every pair, by broadcasting
    return ((diff > 0).float() + 0.5 * (diff == 0).float()).mean().item()

def agree(logits, y):
    """Fraction of rows where the probe's call (logit > 0) matches what the model wrote."""
    return ((logits > 0).float() == y).float().mean().item() if len(y) else float("nan")

def pos_data(d, Hd, lang):
    """Rows of one position for one language: residuals [n, 37, d], label, obeyed flag, question group."""
    sub = d[(d["lang"] == lang) & d["ok"]]
    return dict(H=Hd[torch.tensor(sub.index.to_numpy())], y=torch.tensor(sub["y"].to_numpy(), dtype=torch.float32),
                obeyed=torch.tensor(sub["complied"].to_numpy()), group=sub["group"].tolist())

def run_xpos(lang="ru"):
    P = {"answer": pos_data(pdat, H, lang), "think": pos_data(pdat_t, H_t, lang)}
    groups = sorted(set(P["answer"]["group"]) | set(P["think"]["group"]))
    fold_of = dict(zip(groups, group_folds(groups, K)))                    # one fold per question, shared by both positions
    for p in P.values():
        p["fold"] = torch.tensor([fold_of[g] for g in p["group"]])
    rows = []
    for L in range(H.shape[1]):
        for f in range(K):
            tr = {k: (p["H"][p["fold"] != f, L].float(), p["y"][p["fold"] != f], p["obeyed"][p["fold"] != f]) for k, p in P.items()}
            te = {k: (p["H"][p["fold"] == f, L].float(), p["y"][p["fold"] == f], p["obeyed"][p["fold"] == f]) for k, p in P.items()}
            probes = {"answer": fit_probe(tr["answer"][0], tr["answer"][1]),
                      "think":  fit_probe(tr["think"][0],  tr["think"][1]),
                      "mixed":  fit_probe(torch.cat([tr["answer"][0], tr["think"][0]]), torch.cat([tr["answer"][1], tr["think"][1]]))}
            # request-vs-output test: train on OBEYED rows of both positions, test on held-out DISOBEYED rows
            Xo = torch.cat([tr[k][0][tr[k][2]] for k in P]); yo = torch.cat([tr[k][1][tr[k][2]] for k in P])
            probes["mixed_obeyed"] = fit_probe(Xo, yo)
            for train_on, probe in probes.items():
                for test_on, (Xt, yt, ob) in te.items():
                    z = probe(Xt)
                    rows.append(dict(lang=lang, layer=L, fold=f, train=train_on, test=test_on,
                                     bacc=balanced_acc(z, yt), auc=auc(z, yt), n_dis=int((~ob).sum()),
                                     # only meaningful for the probe that never saw a disobeyed row:
                                     follows=agree(z[~ob], yt[~ob]) if train_on == "mixed_obeyed" else float("nan")))
    return pd.DataFrame(rows)

xp = cached(ACT_DIR / f"c2_probe_xpos_{TAG}.csv", run_xpos)

m = xp.groupby(["train", "test", "layer"])[["bacc", "auc"]].mean()
w = xp[xp["train"] == "mixed_obeyed"].assign(fn=lambda d: d["follows"] * d["n_dis"]).groupby(["test", "layer"])[["fn", "n_dis"]].sum()
follows = w["fn"] / w["n_dis"]                                            # pooled over folds, weighted by fold size

COL = {"answer": "C0", "think": "C1"}                                     # color = position the probe is TESTED at
NAME = {"answer": "answer_start", "think": "think_start"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for test, other in [("answer", "think"), ("think", "answer")]:
    for ax, metric in zip(axes[:2], ["bacc", "auc"]):
        ax.plot(m.loc[(other, test), metric], c=COL[test], ls="--", lw=1.5, label=f"train {NAME[other]} -> test {NAME[test]}")
        ax.plot(m.loc[("mixed", test), metric], c=COL[test], ls="-", lw=1.5, label=f"train both -> test {NAME[test]}")
    n = int(w.loc[test, "n_dis"].iloc[0])
    axes[2].plot(follows.loc[test], c=COL[test], lw=1.5, marker="o", ms=3,
                 label=f"{NAME[test]} (n={n}): asked {'en, wrote ru' if test == 'answer' else 'ru, wrote en'}")
axes[0].set_title("balanced accuracy (threshold logit > 0)")
axes[1].set_title("AUC (does X score above en? ignores a shared shift)")
axes[2].set_title("disobeyed rows: probe agrees with the OUTPUT\n(probe trained on obeyed rows, both positions)")
axes[0].set_ylabel("ru")
for ax in axes:
    ax.axvline(25, c="gray", lw=0.8, ls=":")
    ax.axhline(0.5, c="lightgray", lw=0.8)
    ax.set_xlabel("layer")
    ax.legend(fontsize=8, loc="lower right" if ax is not axes[2] else "upper left")
plt.tight_layout(); plt.show()

In [ ]:
def run_transfer_mixed():
    """No-instruction decision points, scored by a probe trained on BOTH positions' instruction pairs.
    Same question-level folds: a decision item is only scored by probes that never saw its question."""
    rows = []
    for lang in ["ne", "hi", "ru"]:
        P = [pos_data(pdat, H, lang), pos_data(pdat_t, H_t, lang)]
        groups = sorted(set(P[0]["group"]) | set(P[1]["group"]))
        fold_of = dict(zip(groups, group_folds(groups, K)))
        y = torch.cat([p["y"] for p in P])
        fold = torch.tensor([fold_of[g] for p in P for g in p["group"]])
        keys = [k for k in RESID if k[1] == lang]                                  # (item, lang, where)
        kfold = torch.tensor([fold_of[f"mclm_math100:{k[0]}"] for k in keys])
        R = torch.stack([RESID[k] for k in keys])                                  # [40, n_layers+1, d_model]
        for L in range(H.shape[1]):
            X = torch.cat([p["H"][:, L].float() for p in P])                       # both positions stacked
            for f in range(K):
                test = kfold == f
                if not test.any():
                    continue
                z = fit_probe(X[fold != f], y[fold != f])(R[test, L].float())
                for k, zi in zip([k for k, t in zip(keys, test) if t], z.tolist()):
                    rows.append(dict(item=k[0], lang=lang, where=k[2], layer=L, logit=zi, saysX=int(zi > 0)))
    return pd.DataFrame(rows)

trm = cached(ACT_DIR / f"c2_probe_transfer_mixed_{TAG}.csv", run_transfer_mixed)

COL = {"answer_start": "C0", "think_start": "C1"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4), sharey=True)
for ax, lang in zip(axes, ["ne", "hi", "ru"]):
    for where in ["answer_start", "think_start"]:
        d = dec[(dec["lang"] == lang) & (dec["where"] == where)]
        writes = lang if (d["pX"] > d["pL"]).mean() > 0.5 else "en"           # what the model actually writes here
        old = tr[(tr["lang"] == lang) & (tr["where"] == where)].groupby("layer")["saysX"].mean()
        new = trm[(trm["lang"] == lang) & (trm["where"] == where)].groupby("layer")["saysX"].mean()
        ax.plot(old, c=COL[where], ls="--", lw=1, alpha=0.6, label=f"{where}: answer_start-only probe")
        ax.plot(new, c=COL[where], ls="-", lw=1.8, label=f"{where} (writes {writes}): mixed probe")
    ax.set_title(f"{lang}: no instruction, 20 math items")
    ax.axvline(25, c="gray", lw=0.8, ls=":")
    ax.set_xlabel("layer")
    ax.legend(fontsize=7.5, loc="center left")
axes[0].set_ylabel("fraction of items the probe calls X")
plt.tight_layout(); plt.show()


In [ ]:
def run_pooled():
    """ONE probe for all languages. Label = X script (1) vs English (0).
    Trained on: answer_start pairs of ne/hi/ru + think_start pairs of ru ONLY.
    ne/hi think_start rows are never trained on, so the probe can't learn 'ne/hi at think_start = en'.
    Folds by question, shared across languages (ne/hi/ru item 5 are translations of the same question)."""
    A = pdat[pdat["ok"]].assign(pos="answer")
    T = pdat_t[pdat_t["ok"]].assign(pos="think")
    D = pd.concat([A, T])                                        # all pair rows, both positions
    HD = torch.cat([H[torch.tensor(A.index.to_numpy())], H_t[torch.tensor(T.index.to_numpy())]])
    D = D.reset_index(drop=True)                                  # row i of D <-> HD[i]
    in_train = (D["pos"] == "answer") | (D["lang"] == "ru")
    groups = sorted(set(D["group"]))
    fold_of = dict(zip(groups, group_folds(groups, K)))
    fold = torch.tensor(D["group"].map(fold_of).to_numpy())
    y = torch.tensor(D["y"].to_numpy(), dtype=torch.float32)
    trainable = torch.tensor(in_train.to_numpy())

    keys = list(RESID)                                            # natural decision points, all langs
    kfold = torch.tensor([fold_of[f"mclm_math100:{k[0]}"] for k in keys])
    R = torch.stack([RESID[k] for k in keys])

    pair_rows, nat_rows = [], []
    for L in range(H.shape[1]):
        X = HD[:, L].float()
        for f in range(K):
            probe = fit_probe(X[trainable & (fold != f)], y[trainable & (fold != f)])
            te = fold == f                                        # held-out questions: score ALL their rows
            z = probe(X[te])
            for i, zi in zip(te.nonzero().squeeze(1).tolist(), z.tolist()):
                pair_rows.append(dict(row=i, layer=L, logit=zi))
            kt = kfold == f
            for k, zi in zip([k for k, t in zip(keys, kt) if t], probe(R[kt, L].float()).tolist()):
                nat_rows.append(dict(item=k[0], lang=k[1], where=k[2], layer=L, logit=zi))
    pr = pd.DataFrame(pair_rows)
    pr = pr.join(D[["lang", "pos", "target", "y", "complied"]], on="row")   # attach row info
    nat = pd.DataFrame(nat_rows)
    for d in (pr, nat):
        d["saysX"] = (d["logit"] > 0).astype(int)
    return pr, nat

pr, nat = cached(ACT_DIR / f"c2_probe_pooled_{TAG}.pkl", run_pooled)

LCOL = {"ne": "C0", "hi": "C1", "ru": "C2"}
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), sharey=True)
# left: natural decision points, no instruction
for lang in ["ne", "hi", "ru"]:
    for where, ls in [("answer_start", "-"), ("think_start", "--")]:
        d = dec[(dec["lang"] == lang) & (dec["where"] == where)]
        writes = lang if (d["pX"] > d["pL"]).mean() > 0.5 else "en"
        s = nat[(nat["lang"] == lang) & (nat["where"] == where)].groupby("layer")["saysX"].mean()
        axes[0].plot(s, c=LCOL[lang], ls=ls, lw=1.5, label=f"{lang} {where} (writes {writes})")
axes[0].set_title("no instruction: fraction of 20 items the pooled probe calls X")
axes[0].set_ylabel("fraction")
# right: think_start, asked for X but wrote 'Okay' -- rows the probe never trained on
for lang in ["ne", "hi"]:
    d = pr[(pr["lang"] == lang) & (pr["pos"] == "think") & (pr["target"] == lang)]
    s = 1 - d.groupby("layer")["saysX"].mean()                        # agrees with the output (en)
    axes[1].plot(s, c=LCOL[lang], lw=1.5, marker="o", ms=3, label=f"{lang} (n={d['row'].nunique()}): asked {lang}, wrote en")
axes[1].plot(follows.loc["think"], c=LCOL["ru"], lw=1, ls="--", alpha=0.7,
             label=f"ru (n={int(w.loc['think', 'n_dis'].iloc[0])}): asked ru, wrote en  [earlier obeyed-only probe]")
axes[1].set_title("think_start, disobeyed: probe agrees with the OUTPUT (says en)")
for ax in axes:
    ax.axvline(25, c="gray", lw=0.8, ls=":")
    ax.axhline(0.5, c="lightgray", lw=0.8)
    ax.set_xlabel("layer")
    ax.legend(fontsize=8, loc="center left")
plt.tight_layout(); plt.show()